# 13 - Linearity, `ceiling(gain)` and full well

**Purpose.** Run `protocols/05-linearity.md` and publish `ceiling(gain)` - the level at which the
response departs 1% from a straight line, per CFA plane, at the eight gains session 02 measured -
together with the full well in electrons that `g(gain)` turns it into, and the verdict on whether
the bend belongs to the converter or to the pixel. It is the last bench-only row of MISSION's
constants table.

**What it is not for.** Gain, read noise, dark current, PRNU. `g(gain)` is an *input* here, read
from `results/ptc_constants.json` and never re-measured. No pair differencing happens and no
variance is published: the quantity is a **mean level**, and everything this notebook spends its
design on - drift, illumination structure, the offset state - is a thing that ruins a mean.

**Three halves, and they run at different times.** Section 1 reuses session 02's bench
configuration and re-measures it cold, section 2 captures, section 3 reads the frames back off
disk and publishes. Section 3 needs no camera and no kernel state from above it, so a wrong
analysis costs an afternoon and not a bench night.

## 1. The bench, and the five gates

The bench is session 02's - same panel, same sheet count, same grey level, same ROI - and this
notebook **reuses that configuration rather than re-scouting it**. `data/session02/bench.json`
is a record of a configuration, not a published constant, and gate 4 below re-measures every
number in it cold. If the camera or the sheets have been moved since, that file is fiction: re-run
section 1 of `05_ptc.ipynb` first and copy the new `bench.json` here.

Seven gates. Three are session 02's and four are new, and they run in this order because each one
needs the one above it:

| gate | what it settles | new? |
|---|---|---|
| 1 | white balance, verified from the pixels | session 02's |
| 2 | the cooler holds at -10 C | session 02's |
| 3 | **the panel's redraw period**, reported by the page itself | new |
| 4 | the grey level *per gain*, then `t_sat` at that level | session 02's, with the level added |
| 5 | **does the panel flicker** at the shortest rungs the session will shoot | new |
| 6 | is the light steady in wall clock and in exposure length (L31) | new |
| 7 | the illumination map, which chooses the analysis ROI (L09) | new |

Gates 3 to 5 exist because of one thing session 02 could ignore and this session cannot. A photon
transfer curve plots variance against *measured* signal, so a panel that misbehaves moves a point
along the curve. Linearity plots signal against *commanded exposure*, and a screen that redraws
every 16-odd milliseconds is not a steady lamp on that timescale.

In [ ]:
import json
import pathlib
import sys
import time
import urllib.request

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import asi, fits as F, spatial, stats

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session05"
FRAMES = DATA / "frames"
FRAMES.mkdir(parents=True, exist_ok=True)

FULL_SCALE = 4095                    # ADC counts; the units rule in CLAUDE.md
# Four gains, not session 02's eight, and the reason is measured rather than
# chosen: light arrives in redraw pulses, so a rung's error goes as one pulse in
# N, and the first run's rungs scattered 1.2-2.5% against a 1% bend.  Only gains
# whose faintest rung can be made ~300 redraws long at a level the panel can
# reach are shootable.  250, 300 and 450 cannot: 62, 35 and 6 redraws at the
# level floor.  They are out of reach with this light source, and that is
# recorded rather than papered over.
GAINS = [0, 50, 100, 200]
ROI = (1408, 568, 1024, 1024)        # session 02's ROI: even origin and extent, or Bayer shifts
OFFSET = 15                          # project_offset, fixed by session 01
PATCH_SERVER = "http://127.0.0.1:8765"

# The ladder.  Eight geometric rungs to draw the line, sixteen linear ones to
# locate the bend, and the top four are past saturation on purpose.
LINE_RUNGS = [25.0, 31.0, 38.0, 47.0]                          # % of t_sat
BEND_RUNGS = [55.0 + 4.0 * k for k in range(16)]               # 55 .. 115
RUNGS = LINE_RUNGS + BEND_RUNGS
LINE_MAX_PCT = 50.0                  # no rung above this may touch the reference line
MONITOR_PCT = 25.0                   # the drift reference: one between every two ladder frames
BEND_PCT = 1.0                       # the departure that defines the ceiling (L28)

# The panel is a screen, and a screen redraws.  An exposure shorter than one
# redraw does not collect a proportional share of the light -- it collects
# whatever that slice of the cycle happened to be -- and *proportional to
# exposure* is the whole quantity this session measures.  So the ladder is
# scaled per gain to keep even its faintest rung several redraws long.
TARGET_TSAT_S = 22.0                 # the level is chosen to put t_sat here
MIN_RUNG_PERIODS = 150.0             # a gain whose faintest rung is shorter is not shot at all
MIN_LINE_PERIODS = 2.0               # below this a rung may not touch the reference line
LEVEL_FLOOR = 64                     # 25% of white: below it the backlight leak dominates (L07)
LEVEL_CEILING = 255

_bias = json.loads((RESULTS / "bias_constants.json").read_text())
PEDESTAL_FIT = _bias["pedestal_fit"]["value"]
MIN_EXPOSURE = _bias["bias_exposure"]["value"]
HCG = _bias["hcg_threshold_gain"]["value"]

_ptc = json.loads((RESULTS / "ptc_constants.json").read_text())
G_MEASURED = {int(k): v for k, v in _ptc["system_gain"]["value"].items()}
G_ERR = {int(k): v for k, v in _ptc["system_gain"]["uncertainty"].items()}
assert set(GAINS) <= set(G_MEASURED), (
    "every gain here must be one session 02 measured: the gain law's residual is "
    f"{_ptc['gain_law']['value']['residual_pct']}% against its own 1% rule, so g is not "
    "interpolable and a gain with no measured g cannot be turned into electrons")


def amplification(gain):
    """Gain is in 0.1 dB units, so 200 units is exactly a factor of ten."""
    return 10.0 ** (gain / 200.0)


def pedestal_fitted(gain):
    """Session 01's published pedestal at offset 15.  A *prediction* used to
    place a rung; the pedestal every signal is measured against comes from this
    session's own bias block, in section 3."""
    branch = PEDESTAL_FIT["hcg" if gain >= HCG else "lcg"]
    return branch["A"] + branch["B"] * amplification(gain)


def plane_means(mosaic):
    return {k: float(stats.to_adc(v).mean()) for k, v in spatial.split(mosaic).items()}


def panel(path):
    with urllib.request.urlopen(f"{PATCH_SERVER}{path}", timeout=5) as r:
        return json.load(r)


def set_level(level, settle_s=0.8):
    """Drive `grey-patch.html` from here (protocols/patch-server.py)."""
    lvl = level if level == "free" else int(level)
    state = panel(f"/set?level={lvl}")
    time.sleep(settle_s)
    return state


def measure_refresh(seconds=6.0, animated=False):
    """How fast the panel is serving frames, as the page itself reports it.

    `animated` asks the page to animate a 4x4 px dot in its corner.  That is
    not decoration: a display with adaptive refresh serves a *still* page fewer
    frames than the panel drives, so a slow rate with the dot off and a fast
    one with it on means the page was throttled and the panel was not slow.
    Both numbers are recorded, because only the pair is interpretable.
    """
    panel(f"/set?probe={1 if animated else 0}")
    time.sleep(seconds)                       # the page reports every 2 s
    report = panel("/level").get("refresh")
    panel("/set?probe=0")
    if report is None or report["animated"] != animated:
        raise RuntimeError(
            "the page is not reporting its frame rate.  Reload grey-patch.html on the iPad -- "
            "an older copy of the page has no /refresh in it, and this session needs a measured "
            "redraw period rather than an assumed 60 Hz")
    return report


bench = json.loads((ROOT / "data" / "session02" / "bench.json").read_text())
LEVEL_FLUX = {int(k): v for k, v in bench["flux_counts_per_s"].items()}   # a prediction (L07)
print(f"bench reused from session 02: {bench['sheets']} sheets, "
      f"brightness {bench['brightness_pct']}%, measured {bench['measured_on']}")
print(f"{len(RUNGS)} rungs x {len(GAINS)} gains; g measured at all of them")
print("g(gain), e- per ADC count:  " + "  ".join(f"g{g}={G_MEASURED[g]:.4g}" for g in GAINS))
print()
print("grey level is chosen per gain in gate 4, not fixed for the session: it is the LCD "
      "blocking light, so it changes the flux without touching the backlight")

# The panel is driven over HTTP, and every gate below assumes it answers.  Fail
# here, naming the command that starts it, rather than three cells down inside a
# gate that has already opened the camera.
try:
    print("patch server:", panel("/level"))
except OSError as exc:
    raise RuntimeError(
        f"{PATCH_SERVER} is not answering ({exc}).  Start protocols/patch-server.py with the "
        "*base* interpreter, not the venv one -- the venv python is a separate binary with no "
        "inbound firewall rule, and the iPad is on the Public network profile:"
        + chr(10) + "  C:/Users/denis/AppData/Local/Python/pythoncore-3.14-64/python.exe "
        "protocols/patch-server.py"
    ) from None

### Gate 1 - white balance, verified from the pixels (L01)

The camera ships `WB_R=55`, `WB_B=75` and applies them to RAW16 before the data reaches us; the
control reading back as 50 proves only that the control took. The evidence is the modal step
between adjacent distinct values: **16 on all four planes**.

It matters more here than anywhere. White balance multiplies red by 1.10 and blue by 1.50 in
integer arithmetic, so a red plane would appear to saturate at 4095/1.10 and a blue one at
4095/1.50 - three different ceilings on one sensor, which is *exactly* the per-plane result this
session exists to measure. A gate 1 failure here does not add noise; it invents the finding.

In [ ]:
# A dead run is restarted, not resumed -- and the check is the whole session
# directory, not just the frames.  A leftover gate4.csv or panel.json from an
# abandoned attempt would be read back by section 3 and quietly paired with
# tonight's pixels, which is a worse failure than a missing file: the gate
# tables say what exposure each rung *was*, so mixing them mislabels every rung.
leftovers = [f for f in DATA.rglob("*") if f.is_file()]
assert not leftovers, (
    f"{len(leftovers)} file(s) already in {DATA}, first {leftovers[0].name} -- delete the whole "
    "directory and start again.  Section 3 reads the gate tables back and pairs them with the "
    "frames by name, so a survivor from an earlier attempt does not fail loudly, it mislabels")

rig = asi.open_camera()
print("gain range     ", rig.range("Gain"))
print("exposure range ", rig.range("Exposure"), "us")
print("white balance shipped:", rig.get("WB_R"), rig.get("WB_B"))

asi.neutralise_white_balance(rig)
asi.set_roi(rig, *ROI)
asi.configure(rig, gain=100, offset=OFFSET)
BIAS_EXPOSURE = rig.min_exposure_s()          # measured, never assumed

set_level(0)
for _ in range(2):
    asi.capture(rig, BIAS_EXPOSURE)

gate1 = {}
for _ in range(5):
    dark, _ = asi.capture(rig, BIAS_EXPOSURE, imagetyp="DARK")
    for name, plane in spatial.split(dark).items():
        gate1.setdefault(name, []).append(stats.value_step(plane))

steps = {k: sorted(set(v)) for k, v in gate1.items()}
print("modal value step per plane, five frames:", steps)
assert all(v == [16] for v in steps.values()), \
    f"gate 1 FAILED: {steps} -- stop the session, do not correct it later"
print("gate 1 passed")

### Gate 2 - the cooler holds

-10 C, held in band for a continuous 30 seconds before the first frame, judged by the temperature
trend and not by duty cycle. `asi.cool_to` owns the rule and restarts its own clock on any
excursion; every reading is written to disk as it is taken, because the sensor only reports while
we are the ones cooling it.

**The whole session runs in this one kernel.** Closing the camera drops the cooler, and a dead
kernel restarts the cool-down from ambient.

In [ ]:
SETPOINT_C = asi.SETPOINT_C

cool_log = open(DATA / "cooldown.csv", "w", newline="")
cool_log.write("elapsed_s,temp_C,duty_pct" + chr(10))


def show(elapsed, temp, duty):
    cool_log.write(f"{elapsed},{temp},{duty}" + chr(10))
    cool_log.flush()                     # the reading exists nowhere else
    if int(elapsed) % 30 == 0:
        print(f"  {elapsed:6.0f} s  {temp!s:>6} C  {duty:>3}%", flush=True)


try:
    trace = asi.cool_to(rig, SETPOINT_C, log=show)
finally:
    cool_log.close()

temps = [t for _, t, _ in trace if t is not None]
print(f"settled in {trace[-1][0]:.0f} s;  {temps[0]} C -> {temps[-1]} C, minimum {min(temps)} C")
print(f"duty at setpoint {trace[-1][2]}%  <- the headroom this room leaves")

### Gate 3 - the panel's redraw period, measured rather than assumed

"60 Hz" is folklore about a device nobody measured. The page times its own `requestAnimationFrame`
callbacks and reports the median interval, and this reads it back.

**One reading is not enough, and the second one is the point.** A display with adaptive refresh
serves a *still* page fewer frames than the panel actually drives - so a slow rate could mean a
slow panel or a throttled page, and those have opposite consequences. The probe settles it: the
page animates a 4x4 px dot in its corner, which forces a repaint every frame. If the rate rises
with the dot on, the page was being throttled. The dot is black-on-black, four pixels, in a corner
outside any sane ROI, and it is off during every captured frame.

What comes out is `PERIOD_S`, and gate 4 scales the whole ladder against it.

In [ ]:
still = measure_refresh(animated=False)
probed = measure_refresh(animated=True)

print(f"{'':>10} {'period':>10} {'rate':>9} {'p05-p95':>16} {'frames':>7}")
for name, r in (("still", still), ("probed", probed)):
    print(f"{name:>10} {r['period_ms']:9.3f}ms {1000 / r['period_ms']:8.1f}Hz "
          f"{r['p05_ms']:7.3f}-{r['p95_ms']:<7.3f}ms {r['intervals']:7d}")

throttled = probed["period_ms"] < 0.8 * still["period_ms"]
PERIOD_S = min(still["period_ms"], probed["period_ms"]) / 1000.0
jitter = (probed["p95_ms"] - probed["p05_ms"]) / probed["period_ms"]

print()
print("the page was being throttled while still" if throttled else
      "the still page and the probed page are served alike")
print(f"redraw period taken as {PERIOD_S * 1e3:.3f} ms ({1 / PERIOD_S:.1f} Hz) -- the faster of "
      "the two, because the panel cannot drive slower than the frames it delivers")
if jitter > 0.25:
    print(f"WARNING: {100 * jitter:.0f}% spread between the 5th and 95th percentile interval.  "
          "An adaptive-refresh panel that changes rate mid-session makes every rung's phase a "
          "different lottery; gate 5 is the one that decides whether it matters")

### Gate 4 - the grey level per gain, then `t_sat` at that level

`t_sat` is the exposure at which the **brightest** plane fills the headroom above its own
pedestal - brightest, because it is the plane that clips first and the ladder is scaled so that
the clip lands inside it. It is measured at each gain and never extrapolated from one flux and the
0.1 dB law (`light-source.md` item 3).

**The grey level is chosen per gain, and this is the new part.** At one fixed level the ladder
scales with `1/amplification`, so gain 450's faintest rung is 180 times shorter than gain 0's and
lands *inside a single screen redraw*. The rule, in one line: **the brightest level whose faintest
rung still spans `MIN_RUNG_PERIODS` redraws.** Brightest, because level is wall clock; the
constraint is the flicker floor and the objective is time.

Two things make this cheap. Grey level is the LCD blocking light, not the backlight dimming, so it
changes flux without touching the thing that might flicker. And nothing in this session compares
levels *across* gains - the bend is measured per gain, in counts - so a different level per gain
costs the analysis nothing.

It is floored at level 64. Session 02's own curve measured flux 532 at level 160 and 85 at level 0,
so below about a quarter of white the backlight leaking through a black LCD is most of what is
left, and level stops being a control (L07, L08).

The probe lands the brightest plane between 20% and 70% of headroom - below full scale on purpose,
because a probe placed where the bend lives would be scaled by the very non-linearity this session
is here to measure.

In [ ]:
DISCARD = 2                     # frames dropped after a gain change (protocol)
DISCARD_EXPOSURE = 1            # after an exposure change


def measure_flux(gain, guess_s, tries=5):
    """Counts per second on the brightest plane, at one gain, cold."""
    asi.configure(rig, gain=gain, offset=OFFSET)
    for _ in range(DISCARD):
        asi.capture(rig, guess_s, imagetyp="FLAT")

    head, e = FULL_SCALE - pedestal_fitted(gain), guess_s
    for _ in range(tries):
        asi.capture(rig, e, imagetyp="FLAT")               # discard after the change
        mosaic, h = asi.capture(rig, e, imagetyp="FLAT")
        means = plane_means(mosaic)
        plane = max(means, key=means.get)
        signal = means[plane] - pedestal_fitted(gain)
        if 0.2 * head <= signal <= 0.7 * head:
            return signal / h["EXPTIME"], plane, h["EXPTIME"], True
        e = min(max(e * 0.45 * head / max(signal, 1.0), MIN_EXPOSURE), 60.0)
    return signal / h["EXPTIME"], plane, h["EXPTIME"], False


LEVELS = sorted((lv for lv in LEVEL_FLUX if LEVEL_FLOOR <= lv <= LEVEL_CEILING), reverse=True)
REF_LEVEL = bench["grey_level"]
predicted = {int(g): v for g, v in bench["t_sat_predicted_s"].items()}


def choose_level(gain):
    """The level that puts `t_sat` nearest TARGET_TSAT_S at this gain.

    Nearest, not brightest.  Brighter is faster and dimmer is quieter, and the
    first run showed which side that trade falls on: its rungs scattered 1.2-2.5%
    because light arrives in redraw pulses and a short rung catches a partial
    one.  22 s of t_sat puts the faintest rung of this ladder near 300 redraws,
    which is 0.33% of quantisation against a 1% bend.

    Session 02's level-vs-flux curve is a *prediction* used to pick a setting;
    the flux at the chosen level is measured below, and that is the only number
    that reaches the ladder (light-source.md item 3).
    """
    def t_at(lv):
        return predicted[gain] * LEVEL_FLUX[REF_LEVEL] / LEVEL_FLUX[lv]

    return min(LEVELS, key=lambda lv: abs(t_at(lv) - TARGET_TSAT_S))


t_sat, flux, level_of, gate4 = {}, {}, {}, []
print(f"{'gain':>5} {'level':>6} {'plane':>6} {'flux':>11} {'t_sat':>10} "
      f"{'faint rung':>11} {'redraws':>8}")
for g in GAINS:
    level_of[g] = choose_level(g)
    set_level(level_of[g])
    guess = predicted[g] * LEVEL_FLUX[REF_LEVEL] / LEVEL_FLUX[level_of[g]]
    f_g, plane, e, ok = measure_flux(g, max(0.4 * guess, MIN_EXPOSURE))
    flux[g], t_sat[g] = f_g, (FULL_SCALE - pedestal_fitted(g)) / f_g
    faint = RUNGS[0] / 100 * t_sat[g]
    gate4.append({"gain": g, "level": level_of[g], "plane": plane, "probe_s": e, "flux": f_g,
                  "t_sat_s": t_sat[g], "faint_rung_s": faint,
                  "faint_rung_periods": faint / PERIOD_S, "converged": ok})
    print(f"{g:5d} {level_of[g]:6d} {plane:>6} {f_g:11.1f} {t_sat[g]:9.4g}s "
          f"{faint * 1e3:10.2f}ms {faint / PERIOD_S:8.2f}"
          f"{'' if ok else '   <- probe did not converge'}")

ladder = {g: [r / 100 * t_sat[g] for r in RUNGS] for g in GAINS}
monitor_s = {g: MONITOR_PCT / 100 * t_sat[g] for g in GAINS}
faintest = min(min(v) for v in ladder.values())
assert faintest >= MIN_EXPOSURE, (f"faintest rung {faintest * 1e6:.0f} us is under the "
                                  f"{MIN_EXPOSURE * 1e6:.0f} us shutter floor -- the bench is too "
                                  "bright, and no ladder fixes that")

short = [r for r in gate4 if r["faint_rung_periods"] < MIN_RUNG_PERIODS]
print()
assert not short, (
    "these gains cannot reach " + f"{MIN_RUNG_PERIODS:.0f} redraws at any level this panel has: "
    + ", ".join(f"g{r['gain']} at {r['faint_rung_periods']:.0f}" for r in short)
    + ".  Drop them from GAINS rather than shooting a ladder whose rungs are quantised by more "
      "than the bend it is looking for -- the first run did that and published nothing")
print(f"every gain clears {MIN_RUNG_PERIODS:.0f} redraws at its faintest rung; worst is "
      f"{min(r['faint_rung_periods'] for r in gate4):.0f}, which is "
      f"{100 / min(r['faint_rung_periods'] for r in gate4):.2f}% of quantisation against a "
      f"{BEND_PCT}% bend")

N_LADDER, N_BIAS = 3, 10
n_slots = len(RUNGS) * N_LADDER                       # ladder frames per gain
planned = len(GAINS) * (n_slots + n_slots + 1 + N_BIAS)
shutter = sum((N_LADDER + DISCARD_EXPOSURE) * sum(ladder[g])
              + (2 * n_slots + 1) * monitor_s[g] for g in GAINS)
print(f"{planned} frames planned, {planned * ROI[2] * ROI[3] * 2 / 1e9:.2f} GB on C:")
print(f"shutter-open {shutter / 60:.0f} min, plus ~{planned * 0.31 / 60:.0f} min of readout: "
      f"{(shutter + planned * 0.31) / 60:.0f} min in all")

pd.DataFrame(gate4).to_csv(DATA / "gate4.csv", index=False)
(DATA / "panel.json").write_text(json.dumps(
    {"period_s": PERIOD_S, "refresh_still": still, "refresh_probed": probed,
     "throttled": bool(throttled), "level_of_gain": {str(k): v for k, v in level_of.items()},
     "level_floor": LEVEL_FLOOR, "min_rung_periods": MIN_RUNG_PERIODS,
     "min_line_periods": MIN_LINE_PERIODS}, indent=2))
print(f"gate 4 written to {DATA / 'gate4.csv'}, panel state to {DATA / 'panel.json'}")

### Gate 5 - does the panel actually flicker?

Gate 4 made the rungs as long as the light allows. This asks whether the ones that are still short
are usable, and it is the only test here that the camera can run and the page cannot: the page
reports what it is *told* to draw; only the sensor sees what the backlight *did*.

Two questions, at the gain with the shortest rungs:

- **Does the level wobble frame to frame?** Twenty frames at the faintest rung. Their scatter is
  compared against the shot noise of the mean, which is a computable number - `sqrt(S/g + R^2)`
  over the pixel count. Flicker shows up as scatter far above it, because every frame catches a
  different slice of the redraw cycle.
- **Is the level proportional to exposure down there?** Three exposures a factor of two apart,
  all inside the sub-redraw region. Level over exposure should be constant. If it climbs, short
  exposures are being short-changed, and *that* is the failure mode which would masquerade as
  non-linearity at the bottom of the ladder and tilt the reference line.

**A failure here does not stop the session.** It moves the reference line onto the longer rungs
and gets recorded, which is the fourth item of the plan and the reason `MIN_LINE_PERIODS` exists.

In [ ]:
FLICKER_N = 20
FLICKER_STEPS = [1.0, 2.0, 4.0]        # multiples of the faintest rung

def read_noise_counts(gain, offset=OFFSET):
    """Session 01's measured R at the nearest swept gain, in ADC counts."""
    rows = np.genfromtxt(RESULTS / "bias_sweep.csv", delimiter=",", names=True)
    rows = rows[rows["offset"] == offset]
    return float(rows["R_at_offset"][np.argmin(abs(rows["gain"] - gain))])


flicker_gain = min(GAINS, key=lambda g: RUNGS[0] / 100 * t_sat[g] / PERIOD_S)
flicker_plane = next(r["plane"] for r in gate4 if r["gain"] == flicker_gain)
set_level(level_of[flicker_gain])
asi.configure(rig, gain=flicker_gain, offset=OFFSET)
base = RUNGS[0] / 100 * t_sat[flicker_gain]
npix = (ROI[2] // 2) * (ROI[3] // 2)
R_counts = read_noise_counts(flicker_gain)

print(f"gain {flicker_gain}, faintest rung {base * 1e3:.2f} ms = "
      f"{base / PERIOD_S:.2f} redraws, level {level_of[flicker_gain]}")
print()
print(f"{'exposure':>10} {'redraws':>8} {'level':>10} {'measured':>10} {'shot':>10} {'ratio':>7}")

gate5, levels_seen = [], {}
for k in FLICKER_STEPS:
    e = k * base
    for _ in range(DISCARD_EXPOSURE):
        asi.capture(rig, e, imagetyp="FLAT")
    vals = []
    for i in range(FLICKER_N if k == 1.0 else 5):
        mosaic, h = asi.capture(rig, e, imagetyp="FLAT")
        F.write(FRAMES / f"flick_e{k:g}_{i:03d}.fits", mosaic, h)
        vals.append(plane_means(mosaic)[flicker_plane])
    S = float(np.mean(vals)) - pedestal_fitted(flicker_gain)
    measured = float(np.std(vals, ddof=1))
    shot = float(np.sqrt(S / G_MEASURED[flicker_gain] + R_counts ** 2) / np.sqrt(npix))
    levels_seen[k] = S
    gate5.append({"exposure_s": e, "periods": e / PERIOD_S, "signal": S,
                  "scatter": measured, "shot_scatter": shot, "ratio": measured / shot,
                  "n": len(vals)})
    print(f"{e * 1e3:9.2f}ms {e / PERIOD_S:8.2f} {S:10.2f} {measured:10.4f} {shot:10.4f} "
          f"{measured / shot:7.2f}")

prop = [levels_seen[k] / (k * base) for k in FLICKER_STEPS]
print()
print("counts per second at each: " + "  ".join(f"{p:.1f}" for p in prop))
print(f"spread {100 * (max(prop) - min(prop)) / np.mean(prop):.2f}% -- flat means a short "
      "exposure gets its proportional share of the light")
flicker_ok = max(r["ratio"] for r in gate5) < 3.0 and \
    (max(prop) - min(prop)) / np.mean(prop) < 0.01
print("gate 5: " + ("passed -- the short rungs are usable" if flicker_ok else
      "FAILED -- the reference line will be fitted on the long rungs only (section 3)"))
pd.DataFrame(gate5).to_csv(DATA / "gate5_flicker.csv", index=False)

### Gate 6 - is the light steady, in wall clock and in exposure length? (L31)

L31 is the one inherited claim that could invalidate this session outright: two linearity runs
where frame pairs at the same rung should have differed only by noise, and at gain 100 they
differed by **1.79%** against 0.011% at gain 200. An exposure ladder at a fixed light level is
exactly that measurement, so this runs before the ladder and not after it.

Two arms, both at gain 100 - L31's unstable one - and five minutes each:

1. **Drift.** One fixed exposure at 40% of `t_sat`, repeated. Level against wall clock. Session
   01's dark arm already ran this with the light taken out and was flat to 0.00133 +/- 0.254
   counts/min, so anything here is upstream of the sensor.
2. **Exposure length against elapsed time.** A short (10%) and a long (90%) exposure alternating
   throughout. Their normalised fluxes should agree; L31's two gains differed in exposure length
   *and* in elapsed time, so their comparison could not separate the two and this one can.

**A failing gate 6 does not stop the session.** The monitor rungs below divide out drift on any
timescale longer than one rung. What gate 6 decides is whether that correction is a safety net or
load-bearing - and that has to be known before the numbers are read, not after.

In [ ]:
STAB_GAIN = 100
STAB_S = 300.0                  # five minutes per arm
STAB_SHORT_PCT, STAB_LONG_PCT = 10.0, 90.0

asi.configure(rig, gain=STAB_GAIN, offset=OFFSET)
set_level(level_of[STAB_GAIN])


def stability_arm(name, exposures):
    """Shoot `exposures` round-robin for STAB_S seconds, writing every frame.

    Nothing is discarded and nothing is retaken: the arm is a measurement *of*
    frame-to-frame variation, and a filter on it would remove the signal.
    """
    t0, i = time.monotonic(), 0
    for _ in range(DISCARD):
        asi.capture(rig, exposures[0], imagetyp="FLAT")
    while time.monotonic() - t0 < STAB_S:
        e = exposures[i % len(exposures)]
        mosaic, header = asi.capture(rig, e, imagetyp="FLAT")
        F.write(FRAMES / f"stab_{name}_{i:04d}.fits", mosaic, header)
        i += 1
    print(f"  arm {name}: {i} frames in {(time.monotonic() - t0):.0f} s", flush=True)
    return i


n_drift = stability_arm("drift", [monitor_s[STAB_GAIN]])
n_alt = stability_arm("alt", [STAB_SHORT_PCT / 100 * t_sat[STAB_GAIN],
                              STAB_LONG_PCT / 100 * t_sat[STAB_GAIN]])
print(f"gate 6 captured: {n_drift} drift frames, {n_alt} alternating.  Read in section 3.")

### Gate 7 - the illumination map, which chooses the ROI (L09)

L09 says the panel varies **3.8% peak-to-peak across 1024x1024**, so the bright corner saturates
about 4% of exposure before the dim one - smearing a 1% bend over more range than the bend itself.
Central 512 gives 1.25%, central 256 gives 0.53%.

That is a claim about a different bench and it is measured here rather than believed. The rule is
fixed in the protocol: **the smallest box whose peak-to-peak is under 0.5%**, and capture stays at
1024 regardless so the choice can be re-made from the same frames in section 3.

Peak-to-peak is measured on a *plane*, not on the mosaic: the four CFA planes have different
sensitivities and their offsets from each other would otherwise read as illumination structure.

In [ ]:
CROPS = [1024, 512, 256, 128]


def crop_centre(a, n):
    h, w = a.shape
    y, x = ((h - n) // 2) & ~1, ((w - n) // 2) & ~1      # even, or the Bayer phase shifts (L05)
    return a[y:y + n, x:x + n]


def flatness(mosaic, n, side=16):
    """Peak-to-peak of the tile means inside the central `n` box, per plane.

    The tile is a fixed 32x32 mosaic pixels -- 16 a side per plane -- at every
    box, so the shot noise inside one tile does not change as the box does.  Fix
    the tile *count* instead and a small box gets small tiles, the noise in each
    tile mean rises, and the flattest box reads as the worst one.

    One frame, so this is illumination *plus* noise and cannot separate them.
    Section 3 does that from the monitor stacks, and makes the real choice.
    """
    out = {}
    for name, plane in spatial.split(crop_centre(stats.to_adc(mosaic), n)).items():
        k = plane.shape[0] // side
        m = plane[:k * side, :k * side].reshape(k, side, k, side).mean(axis=(1, 3))
        out[name] = float((m.max() - m.min()) / m.mean())
    return out


for _ in range(DISCARD_EXPOSURE):
    asi.capture(rig, 0.5 * t_sat[STAB_GAIN], imagetyp="FLAT")
mosaic, header = asi.capture(rig, 0.5 * t_sat[STAB_GAIN], imagetyp="FLAT")
F.write(FRAMES / "map_000.fits", mosaic, header)

print(f"{'box':>6}  " + "  ".join(f"{p:>8}" for p in spatial.PLANES) + f"  {'worst':>8}   L09 said")
L09_SAID = {1024: 3.8, 512: 1.25, 256: 0.53, 128: None}
worst = {}
for n in CROPS:
    pp = flatness(mosaic, n)
    worst[n] = max(pp.values())
    said = "-" if L09_SAID[n] is None else f"{L09_SAID[n]:.2f}%"
    print(f"{n:>6}  " + "  ".join(f"{100 * pp[p]:7.3f}%" for p in spatial.PLANES)
          + f"  {100 * worst[n]:7.3f}%   {said}")

FLAT_ENOUGH = 0.005
passing = [n for n in CROPS if worst[n] < FLAT_ENOUGH]
first_look = max(passing) if passing else min(CROPS, key=lambda n: worst[n])
print()
print(f"first look says central {first_look}: the largest box under {FLAT_ENOUGH:.1%}, or the "
      "flattest if none passes.  Largest, because among boxes flat enough to trust, the biggest "
      "has the quietest mean")
print(f"a {100 * worst[first_look]:.3f}% spread saturates its bright corner "
      f"{100 * worst[first_look]:.3f}% of exposure early, against a {BEND_PCT}% bend -- and one "
      "frame cannot say how much of that spread is shot noise.  Section 3 decides")
(DATA / "gate7.json").write_text(json.dumps(
    {"peak_to_peak": {str(n): flatness(mosaic, n) for n in CROPS},
     "analysis_box": first_look,
     "rule": f"largest box under {FLAT_ENOUGH}, decided in section 3"}, indent=2))

## 2. The session

**A dead run is restarted, not resumed** - session 01's rule. Delete the whole of
`data/session05/` and go again: the cooler has to settle from scratch anyway, and resumption logic
is code that runs once, in the dark, under time pressure, having never been tested on the case it
exists for.

**The whole directory, not just the frames.** `fits.write` refuses to overwrite, so a surviving
*frame* stops the run loudly. A surviving *gate table* does not: section 3 reads `gate4.csv` and
`panel.json` back and pairs them with the frames by name, so an abandoned attempt's tables would
be silently applied to tonight's pixels and mislabel every rung's exposure. The cell below
therefore refuses to start on any file in the session directory.

**Temperature is enforced by retaking**, as in session 02: a frame whose own header says it was
shot outside the band is not written, and 10 retakes of one slot or a hold past 300 s stops the
session. A retake here can cost a 30-second exposure, so the budget is the same and it binds
harder.

### The grey level, and the monitor rungs

**Each gain shoots at its own grey level**, set at the top of its block and put back after the
bias frames, exactly as gate 4 chose it. The monitors move with it, so every correction below is
in that block's own units and no number crosses a level boundary.

Between every ladder rung, one frame at a fixed 40% of `t_sat`. Each rung is divided by the mean
of the two monitors bracketing it, so **drift on any timescale longer than one rung leaves by
construction** - whatever gate 6 says about its cause. It is also L31's arm 2 running through the
whole session rather than once at the start, and it costs 25 frames per gain.

In [ ]:
FRAME_GAP_S = 0.2               # readout, USB and the file write are what heat the sensor
HOLD_TIMEOUT_S = 300.0          # a hold that never ends is a cooler fault, not a wait
MAX_RETAKES = 10                # per frame slot; more than this is not a transient
BIAS_LEVEL = 0                  # the pedestal block shoots with the panel driven black


def hold_for_temperature(where, exposure_s):
    """Shoot discards until the sensor has been in band for `asi.RECOVER_S`."""
    t0, in_band_since, worst_t = time.monotonic(), None, None
    while True:
        _, header = asi.capture(rig, exposure_s, imagetyp="FLAT")     # discarded on purpose
        time.sleep(FRAME_GAP_S)
        temp, now = header["CCD-TEMP"], time.monotonic()
        worst_t = temp if worst_t is None or (temp is not None and temp > worst_t) else worst_t

        if temp is not None and abs(temp - SETPOINT_C) <= asi.BAND_C:
            in_band_since = now if in_band_since is None else in_band_since
            if now - in_band_since >= asi.RECOVER_S:
                held = now - t0
                print(f"      held {held:.0f} s at {where}, peak {worst_t} C, back at {temp} C",
                      flush=True)
                return held
        else:
            in_band_since = None

        if now - t0 > HOLD_TIMEOUT_S:
            raise TimeoutError(
                f"{HOLD_TIMEOUT_S:.0f} s of holding at {where} and still {temp} C: the cooler is "
                "not keeping up.  Stop the session and check the ambient and the fan -- do not "
                "widen the band")


def capture_frames(n, exposure_s, imagetyp, name, where):
    """`n` in-band frames at one setting, written as `name`_000.fits onwards."""
    retaken, held_s = 0, 0.0
    for i in range(n):
        for _ in range(MAX_RETAKES + 1):
            mosaic, header = asi.capture(rig, exposure_s, imagetyp=imagetyp)
            temp = header["CCD-TEMP"]
            if temp is not None and abs(temp - SETPOINT_C) <= asi.BAND_C:
                break
            retaken += 1
            print(f"    ! {where} frame {i} at {temp} C, retaking", flush=True)
            if temp is not None and temp > SETPOINT_C + asi.BAND_C:
                held_s += hold_for_temperature(where, exposure_s)
            else:
                time.sleep(FRAME_GAP_S)
        else:
            raise RuntimeError(
                f"{MAX_RETAKES} retakes at {where} and still {temp} C.  This is no longer a "
                "transient -- stop the session rather than filling the ladder with frames nobody "
                "can defend")

        F.write(FRAMES / f"{name}_{i:03d}.fits", mosaic, header)
        time.sleep(FRAME_GAP_S)
    return retaken, held_s


def run_gain(gain):
    """One gain: monitor, frame, monitor, frame ... then its own pedestal block.

    **A monitor between every ladder frame, not every rung.** The first run
    bracketed a whole rung -- three frames, twenty-odd seconds -- and the panel
    moves faster than that: its monitor factors stayed inside 1% while the rungs
    they were meant to correct moved by 2%. A correction that is slower than the
    thing it corrects is decoration. Here ladder frame `j` sits between monitors
    `j` and `j+1`, seconds apart, and section 3 divides frame by frame.

    It costs one extra frame per ladder frame, at a quarter of `t_sat` each.
    """
    t0, retaken, held = time.monotonic(), 0, 0.0
    asi.configure(rig, gain=gain, offset=OFFSET)
    set_level(level_of[gain])          # this gain's own level, chosen in gate 4
    for _ in range(DISCARD):
        asi.capture(rig, monitor_s[gain], imagetyp="FLAT")

    slots = [(ri, e) for ri, e in enumerate(ladder[gain]) for _ in range(N_LADDER)]
    last = {}
    for j, (ri, exposure_s) in enumerate(slots):
        for _ in range(DISCARD_EXPOSURE):
            asi.capture(rig, monitor_s[gain], imagetyp="FLAT")
        r, h = capture_frames(1, monitor_s[gain], "FLAT", f"mon_g{gain:03d}_m{j:03d}",
                              f"gain {gain} monitor {j}")
        retaken, held = retaken + r, held + h

        for _ in range(DISCARD_EXPOSURE):
            asi.capture(rig, exposure_s, imagetyp="FLAT")
        i = last.get(ri, -1) + 1
        last[ri] = i
        r, h = capture_frames(1, exposure_s, "FLAT", f"flat_g{gain:03d}_r{ri:02d}_f{i:02d}",
                              f"gain {gain} rung {ri} frame {i} ({exposure_s:.2f} s)")
        retaken, held = retaken + r, held + h
        if i + 1 == N_LADDER:
            print(f"    rung {ri:>2}/{len(RUNGS)}  {RUNGS[ri]:5.1f}%  {exposure_s:8.3f} s  "
                  f"{(time.monotonic() - t0) / 60:5.1f} min", flush=True)

    for _ in range(DISCARD_EXPOSURE):
        asi.capture(rig, monitor_s[gain], imagetyp="FLAT")
    r, h = capture_frames(1, monitor_s[gain], "FLAT", f"mon_g{gain:03d}_m{len(slots):03d}",
                          f"gain {gain} monitor {len(slots)}")     # closes the last bracket
    retaken, held = retaken + r, held + h

    set_level(BIAS_LEVEL)
    for _ in range(DISCARD_EXPOSURE):
        asi.capture(rig, BIAS_EXPOSURE, imagetyp="BIAS")
    r, h = capture_frames(N_BIAS, BIAS_EXPOSURE, "BIAS", f"bias_g{gain:03d}",
                          f"gain {gain} pedestal")
    retaken, held = retaken + r, held + h
    set_level(level_of[gain])

    print(f"  gain {gain}: {retaken} retaken, {held / 60:.1f} min held, "
          f"{(time.monotonic() - t0) / 60:.1f} min", flush=True)
    return retaken, held

### Blocks 1, 2 and 3 - the ladder, its monitors, and the pedestal beside it

| block | gains | frames |
|---|---|---|
| 1 - ladder | the eight | 24 rungs x 3 |
| 2 - monitor | the eight | 25, interleaved rather than blocked |
| 3 - pedestal | the eight | 10, shot adjacent to that gain's ladder |

**Why the pedestal block matters more here than in session 02.** Session 02 published its `g`
against a bias block whose frames sat in two offset states, and session 11 had to repair it. The
repair is free if it is done first: section 3 runs `stats.offset_state` over this block and takes
the **near-state level**, not the raw mean.

**Why block 3 drives the panel black rather than covering the lens.** A cover is a bench
disturbance, and the attenuation is only valid while nobody moves the camera. What still leaks
through a black LCD (L07) is printed below as a measured smallness rather than an assumed zero.

In [ ]:
leak = {g: LEVEL_FLUX[0] * amplification(g) / amplification(bench["scout_gain"])
             * BIAS_EXPOSURE for g in GAINS}
print("light still arriving in a block-3 bias frame, from session 02's level-0 flux:")
print("  " + "  ".join(f"g{g}={leak[g]:.3f}" for g in GAINS) + "  counts")
print(f"worst is {max(leak.values()):.3f} counts against a faintest rung of "
      f"{RUNGS[0] / 100 * (FULL_SCALE - pedestal_fitted(GAINS[-1])):.1f} counts")
print()

t0, retaken, held = time.monotonic(), 0, 0.0
for k, g in enumerate(GAINS, 1):
    print(f"gain {g}  ({k}/{len(GAINS)})", flush=True)
    r, h = run_gain(g)
    retaken, held = retaken + r, held + h
    elapsed = time.monotonic() - t0
    print(f"  == {elapsed / 60:5.1f} min elapsed, "
          f"{elapsed / k * (len(GAINS) - k) / 60:5.1f} min to go", flush=True)

print(f"session: {len(list(FRAMES.glob('*.fits')))} frames, {retaken} retaken, "
      f"{held / 60:.1f} min held, {(time.monotonic() - t0) / 60:.1f} min total")

### Closing down

The cooler is switched off deliberately and the camera closed, and the panel handed back. Let the
sensor warm before unplugging: condensation on a cold sensor is a hardware problem, not a data one.

**Do not move the camera or the sheets until the frame count below is right.** A short session is
recoverable while the bench is still standing and not afterwards.

In [ ]:
rig.set("CoolerOn", 0, verify=False)
rig.close()
set_level("free")

on_disk = sorted(FRAMES.glob("*.fits"))
print("cooler off, camera closed.  Let it reach ambient before unplugging.")
print(f"{len(on_disk)} frames on disk in {FRAMES}, {planned} planned plus gates 4 and 5")
for prefix in ("flat", "mon", "bias", "stab", "map"):
    print(f"  {prefix:>5}: {sum(1 for f in on_disk if f.name.startswith(prefix))}")

## 3. The analysis

**Everything from here reads disk and nothing else.** The frames, `data/session05/gate4.csv`,
`data/session05/gate7.json`, `data/session05/panel.json`, and session 02's `results/ptc_constants.json` are the whole input, so
the analysis can be re-run and corrected without ever costing a bench night.

It runs `protocols/05-linearity.md`'s eight analysis rules, fixed before the data existed:

| rule | what it does | published as |
|---|---|---|
| 1 | everything per CFA plane, never on the frame mean | every row of `linearity_rungs.csv` |
| 2 | pedestal from this session's bias block, near state only | `pedestal` column |
| 3 | every rung divided by the two monitors bracketing it | `monitor_factor`, `signal` |
| 4 | the reference line fitted through the low rungs, through the origin | `line_slope` |
| 5 | `ceiling` = where the departure reaches 1% | `linearity_constants.json` |
| 6 | a rung with any pinned pixel cannot define the bend | `pinned_frac`, `usable` |
| 7 | rules 3-5 re-run at every crop size | `roi` column, and the ROI test |
| 8 | bend levels across gains: the converter or the pixel | the verdict |

In [ ]:
# Section 3 is self-contained on purpose: a fresh kernel can run from here down,
# because the frames and the four files below are the whole input.  Nothing here
# reaches back into a variable the capture half left in memory.
import datetime as dt
import json
import pathlib
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import fits as F, spatial, stats

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session05"
FRAMES = DATA / "frames"

FULL_SCALE = 4095
GAINS = [0, 50, 100, 200]
LINE_RUNGS = [25.0, 31.0, 38.0, 47.0]
BEND_RUNGS = [55.0 + 4.0 * k for k in range(16)]
RUNGS = LINE_RUNGS + BEND_RUNGS
LINE_MAX_PCT, BEND_PCT, MONITOR_PCT = 50.0, 1.0, 25.0
N_LADDER = 3
ROI = (1408, 568, 1024, 1024)
OFFSET = 15
STAB_GAIN = 100

_bias = json.loads((RESULTS / "bias_constants.json").read_text())
_ptc = json.loads((RESULTS / "ptc_constants.json").read_text())
G_MEASURED = {int(k): v for k, v in _ptc["system_gain"]["value"].items()}
G_ERR = {int(k): v for k, v in _ptc["system_gain"]["uncertainty"].items()}
PEDESTAL_FIT, HCG = _bias["pedestal_fit"]["value"], _bias["hcg_threshold_gain"]["value"]


def pedestal_fitted(gain):
    branch = PEDESTAL_FIT["hcg" if gain >= HCG else "lcg"]
    return branch["A"] + branch["B"] * 10.0 ** (gain / 200.0)


RUNGS_CSV = RESULTS / "linearity_rungs.csv"
STABILITY_CSV = RESULTS / "light_stability.csv"
CONSTANTS = RESULTS / "linearity_constants.json"

PLANES = spatial.PLANES
gate7 = json.loads((DATA / "gate7.json").read_text())      # the capture-time first look
gate4 = pd.read_csv(DATA / "gate4.csv")
t_sat = dict(zip(gate4.gain, gate4.t_sat_s))
level_of = dict(zip(gate4.gain, gate4.level))

panel_state = json.loads((DATA / "panel.json").read_text())
PERIOD_S = panel_state["period_s"]

# The flicker floor is gate 5's measurement, not the protocol's guess: the
# shortest exposure whose frame-to-frame scatter came back within FLICKER_MAX of
# the shot noise of the mean.  A rung below it is dropped everywhere -- from the
# line and from the bend search alike, because a rung that did not collect a
# proportional share of the light is not a valid point of either.
# Rule 6 said "any pinned pixel disqualifies a rung", and on a 1024x1024 ROI that
# is too strict to be usable: a handful of hot pixels pin long before the mean
# nears the top code, and the rung that carries the bend is exactly the one they
# disqualify.  The threshold is a fraction whose bias on the plane mean is below
# the thing being measured -- 1% of pixels clipped drags the mean by well under
# 0.1% of signal, and it drags it *down*, so the ceiling reads low rather than high.
PINNED_MAX = 0.01

# What a ceiling has to clear before it is allowed to be one.  A 1% departure is
# only a measurement if the line it departs from is straighter than that, and if
# the rung it happens on is long enough that light arriving in discrete redraw
# pulses cannot fake it -- one pulse in N, so 0.3% wants 333 redraws.
LINE_RESID_MAX = 0.5      # %, worst rung about the fitted line
QUANT_MAX = 0.003         # one redraw period over the crossing rung's exposure

FLICKER_MAX = 3.0
flicker = pd.read_csv(DATA / "gate5_flicker.csv")
clean = flicker[flicker.ratio < FLICKER_MAX]
MIN_PERIODS = float(clean.periods.min()) if len(clean) else float(flicker.periods.max() * 2)
print(f"panel redraw {PERIOD_S * 1e3:.3f} ms")
print(f"gate 5: scatter/shot " +
      "  ".join(f"{r.periods:.2f}->{r.ratio:.1f}x" for r in flicker.itertuples()))
print(f"flicker floor taken as {MIN_PERIODS:.2f} redraws = "
      f"{MIN_PERIODS * PERIOD_S * 1e3:.1f} ms; shorter rungs are dropped everywhere")


def crop_centre(a, n):
    h, w = a.shape
    y, x = ((h - n) // 2) & ~1, ((w - n) // 2) & ~1
    return a[y:y + n, x:x + n]


def frame_row(path, boxes):
    """One frame at several crop sizes: plane mean and pinned fraction each.

    Pinned is `== FULL_SCALE` in ADC counts.  It is counted per plane and per
    box because rule 6 needs it per plane: the plateaus at 25% and then 75% of
    the mosaic are one channel topping out, then three (L12).
    """
    mosaic, header = F.read(path)
    adc = stats.to_adc(mosaic).astype(np.float64)
    out = []
    for n in boxes:
        for name, plane in spatial.split(crop_centre(adc, n)).items():
            out.append({"roi": n, "plane": name, "level": float(plane.mean()),
                        "pinned_frac": float((plane >= FULL_SCALE).mean()),
                        "exptime": float(header["EXPTIME"]),
                        "ccd_temp": header["CCD-TEMP"]})
    return out


BOXES = sorted(int(b) for b in gate7["peak_to_peak"])
print(f"rule 7 re-runs everything at {BOXES}")
print(f"{len(list(FRAMES.glob('*.fits')))} frames to read")

### The analysis box, chosen from the monitor stack (rule 7, L09)

Gate 7 took one frame and printed a first look. This decides, and it does two things that one
frame cannot.

**It measures the noise floor instead of walking into it.** A single flat's tile-to-tile spread is
part illumination and part shot noise, and the noise part *grows* as the box shrinks, because each
tile holds fewer pixels. Judge boxes on that number and the smallest box looks the worst, which is
backwards. So: split each gain-0 monitor stack into odd and even frames, average each, and take
half their difference. That difference contains only noise, and `sqrt(spread^2 - noise^2)` is the
illumination alone. The tile is a fixed **32x32 mosaic pixels** at every box, so nothing changes
underfoot as the box changes.

**And it takes the largest box that passes, not the smallest.** A smaller box is flatter but
noisier: fewer pixels in the plane mean. The flatness test is the constraint - under 0.5%
peak-to-peak - and among boxes that pass it, the biggest is the one with the quietest mean. The
protocol said "smallest" and that was wrong; the rule here is the one that survives being written
down next to its reason.

In [ ]:
TILE_MOSAIC_PX = 32                    # so 16 plane pixels a side, at every box
FLAT_ENOUGH = 0.005


def tile_means(plane, side=TILE_MOSAIC_PX // 2):
    n = plane.shape[0] // side
    return plane[:n * side, :n * side].reshape(n, side, n, side).mean(axis=(1, 3))


def flat_map(paths, box):
    """Illumination spread inside a box, with the shot noise measured and removed."""
    acc = {0: None, 1: None}
    count = {0: 0, 1: 0}
    for i, p in enumerate(paths):
        arr = stats.to_adc(F.read(p)[0]).astype(np.float64)
        acc[i % 2] = arr if acc[i % 2] is None else acc[i % 2] + arr
        count[i % 2] += 1
    a, b = acc[0] / count[0], acc[1] / count[1]
    signal, noise = (a + b) / 2, (a - b) / 2

    out = {}
    for name in spatial.PLANES:
        s = tile_means(spatial.split(crop_centre(signal, box))[name])
        d = tile_means(spatial.split(crop_centre(noise, box))[name])
        p2p = float((s.max() - s.min()) / s.mean())
        floor = float((d.max() - d.min()) / s.mean())
        out[name] = {"p2p": p2p, "noise": floor,
                     "illum": float(np.sqrt(max(p2p ** 2 - floor ** 2, 0.0)))}
    return out


MAP_FRAMES = sorted(FRAMES.glob(f"mon_g{GAINS[0]:03d}_m*.fits"))
flat = {b: flat_map(MAP_FRAMES, b) for b in BOXES}

print(f"{len(MAP_FRAMES)} monitor frames at gain {GAINS[0]}, tiles of "
      f"{TILE_MOSAIC_PX}x{TILE_MOSAIC_PX} mosaic px")
print(f"{'box':>5}  " + "  ".join(f"{p:>8}" for p in spatial.PLANES) +
      f"  {'worst':>8} {'noise':>8}   gate 7 saw")
for b in BOXES:
    worst = max(v["illum"] for v in flat[b].values())
    floor = max(v["noise"] for v in flat[b].values())
    saw = max(gate7["peak_to_peak"][str(b)].values())
    print(f"{b:>5}  " + "  ".join(f"{100 * flat[b][p]['illum']:7.3f}%" for p in spatial.PLANES) +
          f"  {100 * worst:7.3f}% {100 * floor:7.3f}%   {100 * saw:7.3f}%")

passing = [b for b in BOXES if max(v["illum"] for v in flat[b].values()) < FLAT_ENOUGH]
ANALYSIS_BOX = (max(passing) if passing else
                min(BOXES, key=lambda b: max(v["illum"] for v in flat[b].values())))
print()
if passing:
    print(f"analysis box: central {ANALYSIS_BOX} -- the largest under {FLAT_ENOUGH:.1%}, "
          "because among boxes flat enough to trust the biggest has the quietest mean")
else:
    print(f"NO box is under {FLAT_ENOUGH:.1%}: taking the flattest, central {ANALYSIS_BOX}, and "
          "the ceiling below carries that smear -- an illumination spread saturates the bright "
          "corner that fraction of exposure early, against a 1% bend")

### Rule 2 - the pedestal, and the offset state

The pedestal is this session's own block 3, at this gain, **not** session 01's fitted law and not
a bias block from another night: L14's cautionary tale is a dark sitting one count below a bias
shot four hours earlier, which produced a negative dark current.

And it is the **near-state level**, not the block mean. The camera's offset sits in one of two
discrete states separated by a fixed packet of charge at the sense node - about 1.5 e- in low
conversion gain, 0.5 in high - and a bias block that hopped between them has a mean pulled by the
occupancy. `stats.offset_state` is the published classifier for it; session 11 had to apply it to
session 02 after the fact, and doing it first costs one function call.

The occupancy is printed because it is a third night's worth of evidence on how often the far
state is visited, and that is a number two sessions are already arguing about.

In [ ]:
pedestal, occupancy, separation = {}, {}, {}
for g in GAINS:
    files = sorted(FRAMES.glob(f"bias_g{g:03d}_*.fits"))
    levels = {p: [] for p in PLANES}
    for f in files:
        for r in frame_row(f, [ANALYSIS_BOX]):
            levels[r["plane"]].append(r["level"])

    near, far, sep = {}, [], []
    for p in PLANES:
        x = np.asarray(levels[p], float)
        state = stats.offset_state(x)
        # The near state is state 0 -- the populated one -- and the pedestal is
        # its mean.  A group that did not separate comes back all-near, which is
        # the honest answer: no state I can see, at a resolution `separation`
        # None already reports (protocol 04, rule 1).
        near[p] = float(x[~state["far"]].mean())
        far.append(float(state["far"].mean()))
        sep.append(state["separation"])
    pedestal[g] = near
    occupancy[g] = float(np.mean(far))
    separation[g] = [s for s in sep if s is not None]

ped = pd.DataFrame(pedestal).T
ped["far_occupancy"] = pd.Series(occupancy)
ped["state_step"] = pd.Series({g: (np.mean(v) if v else np.nan)
                               for g, v in separation.items()})
ped["fitted_law"] = [pedestal_fitted(g) for g in ped.index]
ped.index.name = "gain"
print(ped.round(4))
print()
print("the fitted_law column is session 01's prediction, not an input: the signal below is "
      "measured against this session's own near-state pedestal")

### Rules 1, 3, 6 and 7 - the rung table

One pass over the ladder, at every crop size, and the only pass: everything below reads this table
rather than the pixels.

**Signal is per plane, per gain, against that gain's near-state pedestal, and every *frame* is
divided by the two monitors that bracket it** - not every rung by the two around the rung. That
was the first run's mistake: the bracket was slower than the wobble it was correcting, so the
monitor factors stayed inside 1% while the rungs moved 2%. A frame here sits seconds from each of
its monitors.

The corrected frames are averaged into the rung afterwards, so `repeat_spread` is the spread of
three *already corrected* frames - which makes it a direct read on whether the correction worked.

In [ ]:
rows = []
for g in GAINS:
    slots = [(ri, i) for ri in range(len(RUNGS)) for i in range(N_LADDER)]
    mon = {}
    for mi in range(len(slots) + 1):
        f = FRAMES / f"mon_g{g:03d}_m{mi:03d}_000.fits"
        for r in frame_row(f, BOXES):
            mon[(mi, r["roi"], r["plane"])] = r["level"] - pedestal[g][r["plane"]]

    grand = {(b, p): float(np.mean([mon[(mi, b, p)] for mi in range(len(slots) + 1)]))
             for b in BOXES for p in PLANES}

    for ri, pct in enumerate(RUNGS):
        files = sorted(FRAMES.glob(f"flat_g{g:03d}_r{ri:02d}_f*.fits"))
        loaded = [(j, frame_row(f, BOXES)) for j, f in
                  ((slots.index((ri, i)), f) for i, f in enumerate(files))]
        for b in BOXES:
            for p in PLANES:
                vals = [(j, r) for j, fr in loaded for r in fr
                        if r["roi"] == b and r["plane"] == p]
                factors = [0.5 * (mon[(j, b, p)] + mon[(j + 1, b, p)]) / grand[(b, p)]
                           for j, _ in vals]
                corrected = [(v["level"] - pedestal[g][p]) / f for (_, v), f in zip(vals, factors)]

                level = float(np.mean([v["level"] for _, v in vals]))
                pinned = float(np.mean([v["pinned_frac"] for _, v in vals]))
                factor = float(np.mean(factors))
                raw = level - pedestal[g][p]
                signal = float(np.mean(corrected))
                spread = float(np.std(corrected, ddof=1))
                exptime = float(np.mean([v["exptime"] for _, v in vals]))
                rows.append({
                    "gain": g, "plane": p, "roi": b, "rung": ri, "rung_pct": pct,
                    "exptime": exptime, "periods": exptime / PERIOD_S,
                    "grey_level": level_of[g], "level": level, "pedestal": pedestal[g][p],
                    "signal_raw": raw, "monitor_factor": factor, "signal": signal,
                    "repeat_spread": spread, "pinned_frac": pinned,
                    "flicker_ok": exptime >= MIN_PERIODS * PERIOD_S,
                    "usable": (pinned <= PINNED_MAX) and (exptime >= MIN_PERIODS * PERIOD_S),
                    "ccd_temp": float(np.mean([v["ccd_temp"] for _, v in vals])),
                })

rungs = pd.DataFrame(rows)
rungs.to_csv(RUNGS_CSV, index=False)
print(f"{len(rungs)} rows written to {RUNGS_CSV}")

mf = rungs[rungs.roi == ANALYSIS_BOX].groupby("gain").monitor_factor
print()
print("monitor factor per gain (1.0 is a panel that did not move):")
print(pd.DataFrame({"min": mf.min(), "max": mf.max(),
                    "span_pct": 100 * (mf.max() - mf.min())}).round(4))

### Rules 4 and 5 - the line, and where the response leaves it

The reference line is fitted through the rungs at or below **50% of `t_sat`** only, and **forced
through the origin in exposure**. A free intercept would absorb exactly the pedestal error that
rule 2 exists to remove, and then report a beautiful straight line through a wrong zero.

`ceiling` is the lowest level whose departure from that line reaches **1%**, interpolated between
the two rungs that bracket it. L28 predicts **3984 counts, 97.3% of the top code**, measured twice
to 0.05% - a prediction to reproduce or refute, never an input.

**Rule 6, and the one place it had to bend.** A clipped rung's mean is compressed by the clip,
which is a different thing from the bend, so clipped rungs are out. But "*any* pinned pixel" is
too strict to be usable on a million-pixel ROI: a handful of hot pixels pin long before the mean
goes near the top code, and at gain 0 the R plane's last rung under that rule sits at 3843 counts
while the rung that carries the bend - 4013 counts, 0.5% of pixels pinned - is thrown away. The
threshold is **1% of pixels**, whose bias on the plane mean is far under the 1% being measured,
and which biases the ceiling *down*: conservative in the direction that matters.

**Not every plane reaches its own ceiling, and that is a fact about the light and not a bug.**
The ladder is scaled to `t_sat` of the *brightest* plane, so under a white-ish source the dimmer
planes top out at a fraction of full scale - at gain 0 the R plane reaches 4095 while B is still
at 1638. A ceiling is published for the planes that got there, and the others are recorded as not
reached rather than given a number. The brightest plane is the one the star-colour constraint
binds on (MISSION), so the thing this session is for survives; what it costs is the per-plane
comparison, and the fix next time is a top rung scaled to the *dimmest* plane.

In [ ]:
def bend(sub):
    """Rules 4 and 5 for one (gain, plane, roi).  Returns a dict, always.

    Two things keep this honest, and the first version of it had neither.

    The line is fitted only on low rungs that `usable` accepts, which now means
    both *no pinned pixel* and *long enough to be trusted* -- a rung shorter than
    the flicker floor measured in gate 5 did not collect a proportional share of
    the light, and letting it in tilts the very line the bend is measured against.

    And the bend is searched for **only among the bend rungs**, above
    `LINE_MAX_PCT`, and only where two consecutive rungs are past the threshold.
    A 1% departure at a 120-count rung is 1.2 counts; on a panel that wobbles by
    more than that, a first-crossing search started at the bottom of the ladder
    finds noise every time and calls it saturation.
    """
    sub = sub.sort_values("exptime")
    low = sub.rung_pct <= LINE_MAX_PCT
    line = sub[low & sub.usable]
    dropped = int((low & ~sub.usable & (sub.pinned_frac <= PINNED_MAX)).sum())
    if len(line) < 4:
        return {"line_slope": np.nan, "ceiling": np.nan, "found": False,
                "n_line": len(line), "line_dropped": dropped, "n_cand": 0,
                "line_resid_pct": np.nan, "quant_pct": np.nan,
                "reason": "too few rungs left to draw a line"}

    k = float((line.signal * line.exptime).sum() / (line.exptime ** 2).sum())
    resid = 100 * (line.signal / (k * line.exptime) - 1.0)
    sub = sub.assign(dep=sub.signal / (k * sub.exptime) - 1.0)

    # candidates: the bend half of the ladder, unclipped and unflickered
    cand = sub[~low & sub.usable].reset_index(drop=True)
    below = (cand.dep <= -BEND_PCT / 100).tolist()
    ceiling, found, at = np.nan, False, None
    for i in range(len(cand)):
        # sustained, not a single dip -- except at the very top of the ladder,
        # where the next rung up is the hard clip and there is nothing to sustain
        # into.  Down here the rungs carry thousands of counts, so a 1% departure
        # is tens of counts and not a noise excursion.
        if below[i] and (i + 1 == len(cand) or below[i + 1]):
            at = i
            break

    if at is not None:
        hi = cand.iloc[at]
        # Two precision gates, and a ceiling that fails either is not a ceiling.
        # The line must be straight enough for a 1% departure to mean something,
        # and the crossing rung must be long enough that light arriving in
        # discrete redraw pulses does not quantise it: the error there goes as
        # one pulse in N, so a 100-redraw rung carries 1% before anything else.
        quant = PERIOD_S / hi.exptime
        why = ("line too noisy" if np.abs(resid).max() > LINE_RESID_MAX else
               "exposure too short" if quant > QUANT_MAX else None)
        if why is not None:
            return {"line_slope": k, "ceiling": np.nan, "found": False,
                    "n_line": len(line), "line_dropped": dropped, "n_cand": len(cand),
                    "line_resid_pct": float(np.abs(resid).max()),
                    "quant_pct": 100 * quant, "reason": why}
        earlier = sub[(sub.exptime < hi.exptime) & sub.usable]
        lo = earlier.iloc[-1] if len(earlier) else None
        if lo is not None and lo.dep > -BEND_PCT / 100:
            w = (-BEND_PCT / 100 - lo.dep) / (hi.dep - lo.dep)
            ceiling = float((lo.signal + lo.pedestal)
                            + w * ((hi.signal + hi.pedestal) - (lo.signal + lo.pedestal)))
        else:
            ceiling = float(hi.signal + hi.pedestal)
        found = True
    elif len(cand):
        # no sustained departure before the clip: the ceiling is the top code,
        # and the last honest rung is the most this ladder can say
        ceiling = float(cand.iloc[-1].signal + cand.iloc[-1].pedestal)

    return {"line_slope": k, "ceiling": ceiling, "found": found,
            "n_line": len(line), "line_dropped": dropped, "n_cand": len(cand),
            "line_resid_pct": float(np.abs(resid).max()),
            "quant_pct": 100 * PERIOD_S / cand.iloc[-1].exptime if len(cand) else np.nan,
            "reason": "ok" if found else "no sustained departure before the clip"}


fits_rows = []
for (g, p, b), sub in rungs.groupby(["gain", "plane", "roi"]):
    row = {"gain": g, "plane": p, "roi": b}
    row.update(bend(sub))
    row["pedestal"] = float(sub.pedestal.iloc[0])
    row["signal_at_bend"] = row["ceiling"] - row["pedestal"]
    row["g_e_per_count"] = G_MEASURED[g]
    row["full_well_e"] = row["signal_at_bend"] * G_MEASURED[g]
    fits_rows.append(row)

bends = pd.DataFrame(fits_rows)
bends.loc[~bends.found, ["ceiling", "signal_at_bend", "full_well_e"]] = np.nan
main = bends[bends.roi == ANALYSIS_BOX]
reached = main[main.found]

print(f"why each (gain, plane) did or did not yield a ceiling, central {ANALYSIS_BOX} box:")
print(main.pivot_table(index="gain", columns="plane", values="reason",
                       aggfunc="first").to_string())
print()
print("worst rung about the fitted line, per gain (the 1% bend has to be read against this):")
print(main.groupby("gain").line_resid_pct.max().round(3).to_string())
print()
if len(reached):
    print(f"{BEND_PCT}% departure, per gain, over the planes that got there:")
    print(reached.groupby("gain").agg(
        planes=("plane", lambda s: "+".join(sorted(s))),
        ceiling=("ceiling", "mean"),
        plane_spread_pct=("ceiling", lambda s: 100 * (s.max() - s.min()) / s.mean()),
        full_well_e=("full_well_e", "mean"),
        line_resid_pct=("line_resid_pct", "max")).round(3))
    print()
    print(f"L28 predicted 3984 counts ({100 * 3984 / FULL_SCALE:.1f}% of the top code); "
          f"measured {reached.ceiling.mean():.1f} over {len(reached)} (gain, plane) fits")
else:
    print("NO ceiling is measurable from this dataset, and that is the result.")
    print(f"A {BEND_PCT}% departure cannot be read off a ladder whose own rungs scatter by more "
          "than that about their own straight line.  Everything else this session measured still "
          "stands -- the redraw period, the flicker floor, the stability trace, the illumination "
          "map, the grey-level table -- and `ceiling` is published as not measured, with the "
          "reason, rather than as a number nobody can defend.")
dropped = main.groupby("gain").line_dropped.max()
if dropped.any():
    print()
    print("rungs dropped for being shorter than "
          f"{MIN_PERIODS:.2f} screen redraws (gate 5's measured floor):")
    print(dropped[dropped > 0].to_string())
    print("their bend is fitted against a shorter lever arm, and that is recorded rather than "
          "hidden -- see line_dropped and n_line in the fits")

### Rule 7 - the ROI test (L09)

L09's rule was *use a small ROI for linearity*, and its reason was that uneven illumination smears
the bend over more range than the bend itself. That is a claim with a measurable consequence: the
bend should move as the box grows, and in one direction - a wider box mixes in corners that
saturate earlier, so the departure arrives sooner and the ceiling comes out **low**.

If the ceiling is flat across boxes, L09's rule is not wrong but it is not load-bearing here
either, and that is worth publishing as plainly as the ceiling is.

In [ ]:
roi_test = pd.DataFrame(index=pd.Index(BOXES, name="roi"))
roi_test["illum_pct"] = [100 * max(v["illum"] for v in flat[b].values()) for b in BOXES]
roi_test["line_resid_pct"] = [bends[bends.roi == b].line_resid_pct.max() for b in BOXES]

if bends.found.any():
    got = bends[bends.found].groupby("roi").ceiling
    roi_test["ceiling"] = got.mean()
    roi_test["n"] = got.size()
    roi_test["vs_analysis_box_pct"] = 100 * (roi_test.ceiling / roi_test.ceiling[ANALYSIS_BOX] - 1)
    print(roi_test.round(4))
    print()
    print(f"the ceiling moves {roi_test.vs_analysis_box_pct.abs().max():.3f}% across boxes from "
          f"{min(BOXES)} to {max(BOXES)} px, against a {BEND_PCT}% bend definition")
    print("L09 predicts the wide box reads low; a flat column says the smearing is not what "
          "limits this measurement")
else:
    roi_test["vs_analysis_box_pct"] = np.nan
    print(roi_test.round(4))
    print()
    print("no box yielded a ceiling, so L09's claim is untested here.  What the columns do show "
          "is the ordering L09 predicted -- illumination spread rises with box size - and that "
          "the line residual barely moves with it, which says the box is not what limits this "
          "measurement.  Something common to every box is, and gate 6 names it.")

### Rule 8 - the converter, or the pixel

The one structural question this session can answer. A bend that belongs to the **ADC** sits at a
fixed *level* in counts, at every gain and on every plane. A bend that is the **pixel well
filling** sits at a fixed *charge*, so its level in counts is `Q/g` and rises steeply with gain -
until it runs into the top code and the ADC binds instead.

Both are published, and the verdict is whichever is flat. Per-plane spread is the yardstick: L12
found all four planes bending at one level to 1.6%, which is what made "the converter bends, not
the pixel" a reading rather than a guess.

In [ ]:
per_gain = reached.groupby("gain").agg(
    ceiling=("ceiling", "mean"),
    ceiling_sd=("ceiling", "std"),
    n_planes=("plane", "size"),
    plane_spread_pct=("ceiling", lambda s: 100 * (s.max() - s.min()) / s.mean()),
    signal_at_bend=("signal_at_bend", "mean"),
    full_well_e=("full_well_e", "mean"))
per_gain["pct_of_top_code"] = 100 * per_gain.ceiling / FULL_SCALE
per_gain["Q_over_g_counts"] = per_gain.full_well_e / pd.Series(G_MEASURED).loc[per_gain.index]

if len(per_gain) >= 2:
    flat_in_counts = float(100 * (per_gain.ceiling.max() - per_gain.ceiling.min())
                           / per_gain.ceiling.mean())
    flat_in_charge = float(100 * (per_gain.full_well_e.max() - per_gain.full_well_e.min())
                           / per_gain.full_well_e.mean())
    plane_spread = float(per_gain.plane_spread_pct.max())
    verdict = "converter" if flat_in_counts < flat_in_charge else "pixel well"
    print(per_gain.round(3))
    print()
    print(f"ceiling varies {flat_in_counts:.2f}% across gains in ADC counts")
    print(f"full well varies {flat_in_charge:.2f}% across gains in electrons")
    print(f"worst per-plane spread at one gain: {plane_spread:.2f}%")
    print()
    print(f"verdict: the bend follows the {verdict} -- whichever is flat is what bends")
else:
    flat_in_counts = flat_in_charge = plane_spread = float("nan")
    verdict = None
    print(f"{len(per_gain)} gain(s) yielded a ceiling; the converter-or-pixel question needs at "
          "least two and is not answered by this dataset.  It is published as null, not guessed.")

### Gate 6 read back - the L31 stability trace

Two arms, read off disk. The drift arm is a slope in counts per minute and a frame-to-frame
scatter; the alternating arm is the same scatter computed separately for the short and the long
exposure, plus the ratio of their normalised fluxes.

**What each outcome does.** A flat drift arm from cold **deletes item 1 of `light-source.md`** on
that item's own instruction - a ten-minute warm-up is a ritual once its reason is falsified. A
drift that is real is published beside the ceiling, and every future session shooting second-long
exposures at a fixed light level carries monitor rungs. And if the two arms disagree - steady in
wall clock, unsteady in exposure length - then it is not the panel at all, it is something
exposure-length dependent inside the camera, and that outranks the ceiling as a finding.

In [ ]:
def arm_table(pattern):
    out = []
    for f in sorted(FRAMES.glob(pattern)):
        r = [x for x in frame_row(f, [ANALYSIS_BOX]) if x["plane"] == "G1"][0]
        _, header = F.read(f)
        out.append({"file": f.name, "t": header["DATE-OBS"], "exptime": r["exptime"],
                    "level": r["level"] - pedestal[STAB_GAIN]["G1"]})
    d = pd.DataFrame(out)
    d["minutes"] = (pd.to_datetime(d.t) - pd.to_datetime(d.t).iloc[0]).dt.total_seconds() / 60
    d["flux"] = d.level / d.exptime
    return d


drift = arm_table("stab_drift_*.fits")
alt = arm_table("stab_alt_*.fits")

slope, _ = np.polyfit(drift.minutes, drift.level, 1)
drift_pct = 100 * drift.level.std(ddof=1) / drift.level.mean()
print(f"drift arm: {len(drift)} frames over {drift.minutes.max():.1f} min")
print(f"  slope {slope:+.4f} counts/min ({100 * slope / drift.level.mean():+.4f} %/min), "
      f"frame-to-frame scatter {drift_pct:.3f}%")
print(f"  session 01's dark arm: -0.00133 +/- 0.254 counts/min, with the light taken out")

alt["arm"] = np.where(alt.exptime > alt.exptime.median(), "long", "short")
by_arm = alt.groupby("arm").agg(n=("flux", "size"), exptime=("exptime", "mean"),
                                flux=("flux", "mean"),
                                scatter_pct=("flux", lambda s: 100 * s.std(ddof=1) / s.mean()))
print()
print(by_arm.round(4))
print(f"  long/short flux ratio {by_arm.flux['long'] / by_arm.flux['short']:.4f} "
      "(1.0 means exposure length does not change the measured flux)")
print()
print(f"L31's contrast to beat: 1.79% at gain 100 against 0.011% at gain 200")

stability = pd.concat([drift.assign(arm="drift"), alt], ignore_index=True)
stability.to_csv(STABILITY_CSV, index=False)
print(f"written to {STABILITY_CSV}")

### Publishing

Two CSVs are already written - `linearity_rungs.csv` (every rung, plane and crop size) and
`light_stability.csv` (gate 6's two arms). This cell writes the scalars with their provenance.

`ceiling` is published **per gain and per plane**, because the star-colour constraint binds on the
brightest plane and a single number would hide that. `full_well_e` carries `g(gain)`'s own
uncertainty into its own: the ceiling is a level this session measured, the electrons are session
02's scale applied to it, and a reader is entitled to know which half of the product moved.

In [ ]:
on_disk = sorted(FRAMES.glob("*.fits"))
measured_on = str(F.read(on_disk[0])[1]["DATE-OBS"])[:10]
n_frames = len(on_disk)


def constant(value, unit, uncertainty, note):
    return {"value": value, "unit": unit, "uncertainty": uncertainty,
            "source_frames": n_frames, "measured_on": measured_on,
            "notebook": "13_linearity.ipynb", "note": note}


ceiling_by_gain = {int(g): round(float(v), 2) for g, v in per_gain.ceiling.items()}
well_by_gain = {int(g): round(float(v), 1) for g, v in per_gain.full_well_e.items()}
well_err = {int(g): round(float(per_gain.full_well_e[g] * G_ERR[g] / G_MEASURED[g]), 1)
            for g in per_gain.index}

constants = {
    "ceiling": constant(
        ceiling_by_gain, "ADC counts",
        {int(g): (None if not np.isfinite(v) else round(float(v), 2))
         for g, v in per_gain.ceiling_sd.items()},          # null: one plane, no scatter to quote
        f"rule 5 of protocols/05-linearity.md: the level at which the response departs "
        f"{BEND_PCT}% from a line fitted through the rungs below {LINE_MAX_PCT}% of t_sat and "
        f"forced through the origin.  Mean over the four CFA planes in the central "
        f"{ANALYSIS_BOX} box; per-plane values are in linearity_rungs.csv and the fits per "
        f"(gain, plane, roi) are reproducible from it.  Uncertainty is the per-plane scatter.  "
        f"L28 predicted 3984 counts, 97.3% of the top code"),
    "ceiling_per_plane": constant(
        {int(g): {p: (None if not np.isfinite(v) else round(float(v), 2))
                  for p, v in s.set_index("plane").ceiling.items()}
         for g, s in main.groupby("gain")},
        "ADC counts", None,
        "the constraint binds per plane: sky flux differs per CFA channel, so the exposure "
        "floor is set by the dimmest plane and the clipping ceiling by the brightest (MISSION).  "
        "null means that plane yielded no ceiling -- linearity_rungs.csv carries the rungs and "
        "the fits carry the reason, which is usually that a ladder scaled to the brightest "
        "plane never takes the dimmer ones near their own saturation"),
    "not_measured": constant(
        {f"{int(r.gain)}/{r.plane}": r.reason for r in main.itertuples() if not r.found},
        "reason per (gain, plane)", None,
        "every fit that did not yield a ceiling, and why.  A published null with a reason is the "
        "point of this entry: the alternative is a number that looks like a measurement"),
    "full_well": constant(
        well_by_gain, "e-", well_err,
        "(ceiling - pedestal) x g(gain), with g consumed from ptc_constants.json and never "
        "re-measured here.  The uncertainty is g's alone: the ceiling's own scatter is the "
        "separate ceiling entry, and the two are not independent enough to add"),
    "bend_follows": constant(
        verdict, "one of: converter, pixel well", None,
        f"rule 8.  The ceiling varies {flat_in_counts:.2f}% across gains in ADC counts and the "
        f"full well varies {flat_in_charge:.2f}% in electrons; whichever is flat is what bends.  "
        f"Worst per-plane spread at one gain is {plane_spread:.2f}%, against L12's 1.6%.  "
        f"null means fewer than two gains yielded a ceiling, so the question was not answered"),
    "roi_sensitivity": constant(
        {int(b): (None if not np.isfinite(v) else round(float(v), 4))
         for b, v in roi_test.vs_analysis_box_pct.items()},
        "% change in ceiling against the analysis box", None,
        f"rule 7 / L09.  The analysis box is the central {ANALYSIS_BOX}: the *largest* whose "
        f"illumination spread is under 0.5%, measured on the gain-{GAINS[0]} monitor stack with "
        f"the shot-noise floor subtracted (odd frames against even).  Largest, because among "
        f"boxes flat enough to trust the biggest has the quietest mean.  L09 predicted 3.8% at "
        f"1024, 1.25% at 512 and 0.53% at 256, and claimed a wide box reads the ceiling low"),
    "flicker_floor": constant(
        round(MIN_PERIODS, 3), "screen redraws", None,
        f"the shortest exposure gate 5 found within {FLICKER_MAX}x of the shot noise of the mean, "
        f"in units of the {PERIOD_S * 1e3:.1f} ms redraw period.  Every rung below it is dropped "
        f"from the line *and* from the bend search: a frame that caught part of a redraw cycle "
        f"collected whatever that slice was, not a share proportional to its length"),
    "panel_redraw_period": constant(
        round(PERIOD_S * 1e3, 4), "ms",
        round(panel_state["refresh_probed"]["p95_ms"]
              - panel_state["refresh_probed"]["p05_ms"], 4),
        f"gate 3: the page timed its own requestAnimationFrame callbacks and reported the median "
        f"interval, still and with a 4x4 px corner dot animating.  "
        f"{'The still page was being throttled, so the probed rate is the panel' if panel_state['throttled'] else 'Still and probed agree'}"
        f".  It is the rate frames are *served*, which bounds the panel from below and is not a "
        f"reading of the backlight -- what the backlight does is gate 5, measured with the camera"),
    "sub_redraw_flicker": constant(
        {"scatter_over_shot": round(float(flicker.ratio.max()), 3),
         "flux_spread_pct": round(float(100 * (flicker.signal / flicker.exposure_s).pipe(
             lambda s: (s.max() - s.min()) / s.mean())), 3),
         "shortest_rung_periods": round(float(flicker.periods.min()), 3)},
        "dimensionless", None,
        "gate 5: at the gain with the shortest rungs, frame-to-frame scatter against the shot "
        "noise of the mean, and counts per second across three exposures a factor of two apart "
        "inside the sub-redraw region.  A screen redraws, so an exposure shorter than one redraw "
        "collects whatever slice of the cycle it caught rather than a proportional share -- which "
        "would masquerade as non-linearity at the bottom of the ladder.  Ratio near 1 and a flat "
        "flux mean the short rungs are honest"),
    "grey_level_per_gain": constant(
        {int(g): int(v) for g, v in level_of.items()}, "grey level 0-255", None,
        f"gate 4: the brightest level whose faintest rung still spans "
        f"{panel_state['min_rung_periods']:.0f} screen redraws, floored at "
        f"{panel_state['level_floor']} where the backlight leak takes over (L07).  Grey level is "
        f"the LCD blocking light, not the backlight dimming, so it changes flux without touching "
        f"what might flicker.  Nothing in this session compares levels across gains"),
    "light_drift": constant(
        round(float(slope), 5), "ADC counts per minute", round(float(drift.level.std(ddof=1)), 4),
        f"gate 6 arm 1 (L31): {len(drift)} frames at gain {STAB_GAIN} over "
        f"{drift.minutes.max():.1f} minutes at a fixed exposure, panel warm.  Session 01's dark "
        f"arm measured -0.00133 +/- 0.254 counts/min with the light taken out, so anything here "
        f"is upstream of the sensor"),
    "flux_vs_exposure_length": constant(
        round(float(by_arm.flux["long"] / by_arm.flux["short"]), 5),
        "ratio of counts/s at 90% t_sat to counts/s at 10%", None,
        f"gate 6 arm 2 (L31): the arm the retired project could not run, because their two gains "
        f"differed in exposure length and in elapsed time at once.  1.0 means the measured flux "
        f"does not depend on how long the shutter was open.  Scatter: "
        f"{by_arm.scatter_pct['short']:.3f}% short, {by_arm.scatter_pct['long']:.3f}% long, "
        f"against L31's 1.79% at this gain"),
}

with open(CONSTANTS, "w") as fh:                 # the repo's idiom for a published JSON
    json.dump(constants, fh, indent=2)
    fh.write(chr(10))
print(f"written {CONSTANTS}")
for k, v in constants.items():
    print(f"  {k}: {json.dumps(v['value'])[:90]}")
print()
print("Next: 14_linearity_read.ipynb reads these files back and explains them.  It measures "
      "nothing and writes nothing, and if it ever disagrees with results/, results/ is right.")